## HiMCM26 — three-tier entropy-weighted MCDA

```
HOW THE MODEL WORKS
-------------------
A three-tier funnel: countries -> regions -> cities. Each tier is scored with
entropy-weighted multi-criteria decision analysis (MCDA):

  1. Normalize   each criterion to [0, 1] with min-max, flipped for "cost"
                 criteria, so that 1 is always best.
  2. Weight      each criterion by how much it separates the places (entropy
                 method). No human chooses the weights; the data decides.
  3. Score       each place: the weighted average of its normalized values,
                 using only the criteria it has data for.
  4. Screen      keep places with enough data coverage, take the top N, and
                 search only inside the survivors at the next tier.
  5. Sensitivity change one input for one place and watch its score and rank
                 move, with the weights held fixed.

Comment conventions:
  "# CHANGED:" a fix or improvement made during review, and why.
  Other comments explain what the code does and why it is written that way.
```

In [ ]:
# CELL 1 — IMPORT LIBRARIES AND SET PROJECT FOLDERS

from pathlib import Path
import zipfile                     # CHANGED: removed unused `import io`.
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# CHANGED: `display` exists only inside Jupyter/Colab. This fallback lets
# `python himcm26.py` run too; without it the script stops with NameError.
try:
    display
except NameError:
    def display(obj):
        print(obj)

# Every path is built from the working directory, so the project can be moved
# without editing anything. (In Colab that directory is /content.)
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"

REGISTRY_DIR = DATA_DIR / "geographic_registry"   # GeoNames lists of places
INPUT_DIR = DATA_DIR / "input"                    # your indicator CSVs
TIER1_DIR = INPUT_DIR / "tier1_country"
TIER2_DIR = INPUT_DIR / "tier2_region"
TIER3_DIR = INPUT_DIR / "tier3_city"
OUTPUT_DIR = DATA_DIR / "output"                  # scores, shortlists, weights

# CHANGED: one tier -> folder mapping, defined once and used everywhere.
# (It used to be written out separately in CELL 3 and CELL 7.)
TIER_FOLDERS = {
    "tier1_country": TIER1_DIR,
    "tier2_region": TIER2_DIR,
    "tier3_city": TIER3_DIR
}

# exist_ok=True makes this safe to rerun; parents=True creates data/ and input/.
for folder in [REGISTRY_DIR, *TIER_FOLDERS.values(), OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Place your indicator CSV files in:")
print("Tier 1:", TIER1_DIR)
print("Tier 2:", TIER2_DIR)
print("Tier 3:", TIER3_DIR)

In [ ]:
# CELL 2 — NESTED CRITERIA DICTIONARY
# Structure: tier -> category -> criterion_key -> metadata.
# The model only reads "direction":
#   "benefit" = higher is better, "cost" = lower is better.
# Categories are for readability only; they don't affect the weights.
# Each criterion_key is also the file name: data/input/<tier>/<criterion_key>.csv

CRITERIA = {
    "tier1_country": {
        "healthcare": {
            "physicians_per_1000": {
                "direction": "benefit",
                "unit": "physicians per 1,000 people",
                "description": "Availability of physicians at the country level"
            },
            "health_expenditure_per_capita_ppp": {
                "direction": "benefit",
                "unit": "international dollars per person",
                "description": "Health expenditure per person, PPP-adjusted"
            }
        },
        "education": {
            "secondary_completion_rate": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Share of the relevant population completing secondary education"
            },
            "tertiary_education_rate": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Share of adults with tertiary education"
            }
        },
        "economics_and_opportunity": {
            "gdp_per_capita_ppp": {
                "direction": "benefit",
                "unit": "international dollars per person",
                "description": "GDP per capita adjusted for purchasing power"
            },
            "unemployment_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "National unemployment rate"
            },
            "poverty_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "Share of population below a consistently defined poverty threshold"
            }
        },
        "family_and_social_conditions": {
            "median_household_income_ppp": {
                "direction": "benefit",
                "unit": "international dollars",
                "description": "Median household disposable income, PPP-adjusted"
            },
            "housing_cost_burden": {
                "direction": "cost",
                "unit": "percent",
                "description": "Share of household income spent on housing"
            }
        }
    },

    "tier2_region": {
        "regional_healthcare": {
            "regional_hospital_beds_per_1000": {
                "direction": "benefit",
                "unit": "beds per 1,000 people",
                "description": "Hospital bed availability in the region"
            },
            "regional_physicians_per_1000": {
                "direction": "benefit",
                "unit": "physicians per 1,000 people",
                "description": "Physician availability in the region"
            }
        },
        "regional_economics": {
            "regional_unemployment_rate": {
                "direction": "cost",
                "unit": "percent",
                "description": "Unemployment rate for the administrative region"
            },
            "regional_median_income_ppp": {
                "direction": "benefit",
                "unit": "international dollars",
                "description": "Regional median disposable income, adjusted for purchasing power"
            },
            "regional_housing_cost_burden": {
                "direction": "cost",
                "unit": "percent",
                "description": "Regional housing costs relative to household income"
            }
        },
        "safety_and_environment": {
            "regional_homicide_rate": {
                "direction": "cost",
                "unit": "per 100,000 people",
                "description": "Homicides per 100,000 residents; use a consistent definition"
            },
            "regional_pm25": {
                "direction": "cost",
                "unit": "micrograms per cubic meter",
                "description": "Annual mean fine particulate matter concentration"
            },
            "regional_green_space_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Population with access to green space under one consistent definition"
            }
        }
    },

    "tier3_city": {
        "local_access": {
            "city_healthcare_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Residents within a defined travel time of primary healthcare"
            },
            "public_transport_access": {
                "direction": "benefit",
                "unit": "percent",
                "description": "Residents within a defined walk distance of public transport"
            }
        },
        "local_safety_and_environment": {
            "city_homicide_rate": {
                "direction": "cost",
                "unit": "per 100,000 people",
                "description": "City homicide rate using a consistent reporting definition"
            },
            "city_pm25": {
                "direction": "cost",
                "unit": "micrograms per cubic meter",
                "description": "Annual mean city-level PM2.5 concentration"
            },
            "city_green_space_per_capita": {
                "direction": "benefit",
                "unit": "square meters per person",
                "description": "Public green space per resident"
            }
        },
        "amenities_and_diversity": {
            "restaurants_per_10000": {
                "direction": "benefit",
                "unit": "restaurants per 10,000 people",
                "description": "Restaurant availability, ideally from a consistent POI source"
            },
            "cultural_venues_per_10000": {
                "direction": "benefit",
                "unit": "venues per 10,000 people",
                "description": "Museums, theatres, libraries, and cultural venues per capita"
            },
            "diversity_index": {
                "direction": "benefit",
                "unit": "0 to 1",
                "description": "A documented demographic diversity index"
            }
        }
    }
}

# Check the dictionary now, so a typo fails here with a clear message instead
# of producing a wrong score later.
for tier_name, categories in CRITERIA.items():
    # CHANGED: an unknown tier name would otherwise fail later with a bare KeyError.
    assert tier_name in TIER_FOLDERS, f"No input folder defined for tier {tier_name}"
    seen_keys = set()
    for category_name, variables in categories.items():
        for criterion, details in variables.items():
            assert details["direction"] in ["benefit", "cost"], (
                f"Invalid direction for {criterion}"
            )
            # CHANGED: flatten_criteria() merges all categories into one dict.
            # A key reused in two categories would overwrite the first one
            # without any error.
            assert criterion not in seen_keys, (
                f"Duplicate criterion key '{criterion}' in {tier_name}"
            )
            seen_keys.add(criterion)

print("Criteria successfully loaded.")
for tier_name, categories in CRITERIA.items():
    all_keys = [key for group in categories.values() for key in group]
    print(f"{tier_name}: {len(all_keys)} criteria -> {all_keys}")

In [ ]:
# CELL 3 — CREATE BLANK CSV TEMPLATES FOR EVERY CRITERION
# Writes one empty CSV (headers only) per criterion for you to fill in.
# The loader needs only geo_id and value. year, source and quality are for
# record-keeping, and year also picks the newest row when a place appears twice.
# CHANGED: this cell appeared twice, word for word; the second copy was removed.

REQUIRED_COLUMNS = ["geo_id", "value", "year", "source", "quality"]

def create_csv_templates():
    for tier_name, categories in CRITERIA.items():
        folder = TIER_FOLDERS[tier_name]   # CHANGED: use the shared mapping.

        for category_name, variables in categories.items():
            for criterion_key in variables:
                file_path = folder / f"{criterion_key}.csv"
                # Never overwrite: rerunning must not erase data you entered.
                if not file_path.exists():
                    # CHANGED: build the full column list first, rather than
                    # adding columns to an empty DataFrame afterwards. The
                    # parent IDs only help the person filling the file; rows
                    # are matched on geo_id alone.
                    columns = list(REQUIRED_COLUMNS)
                    if tier_name != "tier1_country":
                        columns.append("country_id")
                    if tier_name == "tier3_city":
                        columns.append("region_id")
                    pd.DataFrame(columns=columns).to_csv(file_path, index=False)

create_csv_templates()
print("Blank criterion CSV templates created (existing files were not overwritten).")

In [ ]:
# CELL 4 — DOWNLOAD COUNTRY, REGION, AND CITY REGISTRIES
# GeoNames supplies the official list of places and IDs at each tier:
#   countries -> ISO2 code ("US"), regions -> "US.MI", cities -> numeric geoname ID.
# Your indicator CSVs must use these same IDs in their geo_id column.

GEONAMES_BASE = "https://download.geonames.org/export/dump"

def download_if_missing(url, destination):
    "Download once and reuse the saved copy afterwards (cities1000.zip is ~10 MB)."
    destination = Path(destination)
    if not destination.exists():
        print("Downloading:", url)
        response = requests.get(url, timeout=180)
        response.raise_for_status()          # stop on 404/500 instead of saving an error page
        # CHANGED: write to a temporary ".part" file, then rename it. If the
        # download is cut off, no half-written file is left at `destination`
        # to be skipped as "already downloaded" and fail later as a corrupt zip.
        temporary = destination.with_name(destination.name + ".part")
        temporary.write_bytes(response.content)
        temporary.replace(destination)
    return destination

country_file = download_if_missing(
    f"{GEONAMES_BASE}/countryInfo.txt",
    REGISTRY_DIR / "countryInfo.txt"
)
region_file = download_if_missing(
    f"{GEONAMES_BASE}/admin1CodesASCII.txt",
    REGISTRY_DIR / "admin1CodesASCII.txt"
)
city_zip_file = download_if_missing(
    f"{GEONAMES_BASE}/cities1000.zip",
    REGISTRY_DIR / "cities1000.zip"
)

print("Registry source files are ready.")

In [ ]:
# CELL 5 — READ THE GEOGRAPHIC REGISTRIES
# Turns the raw GeoNames files into three tidy tables that share one layout:
# geo_id, place_name, and parent IDs (country_id, region_id) that link the tiers.

# CHANGED: by default pandas reads the text "NA" as a missing value. "NA" is
# Namibia's ISO2 code, so Namibia and all its cities were dropped (251 countries
# instead of 252). With these options only truly empty fields count as missing.
NA_SAFE = {"keep_default_na": False, "na_values": [""]}

# GeoNames countryInfo fields (lines starting with "#" are comments and are skipped).
country_columns = [
    "ISO2", "ISO3", "ISO_numeric", "fips", "country_name",
    "capital", "area_sq_km", "population", "continent",
    "tld", "currency_code", "currency_name", "phone",
    "postal_code_format", "postal_code_regex", "languages",
    "geonameid", "neighbours", "equivalent_fips"
]

countries = pd.read_csv(
    country_file, sep="\t", comment="#", header=None,
    names=country_columns, dtype={"ISO2": str, "ISO3": str},
    **NA_SAFE                                            # CHANGED: see NA_SAFE.
)

# First-level administrative regions: code, name, ASCII name, GeoNames ID.
regions = pd.read_csv(
    region_file, sep="\t", header=None,
    names=["region_id", "region_name", "region_ascii", "region_geoname_id"],
    dtype={"region_id": str},
    **NA_SAFE                                            # CHANGED: see NA_SAFE.
)
# Region codes look like "US.MI"; the part before the dot is the parent country.
regions["country_id"] = regions["region_id"].str.split(".").str[0]

# The GeoNames cities1000 file has 19 tab-separated columns.
city_columns = [
    "geoname_id", "city_name", "ascii_name", "alternate_names",
    "latitude", "longitude", "feature_class", "feature_code",
    "country_id", "alternate_country_codes", "admin1_code",
    "admin2_code", "admin3_code", "admin4_code", "population",
    "elevation", "dem", "timezone", "modification_date"
]
# CHANGED (optimization): parse only the 9 columns the model uses. The big
# alternate_names text column alone holds most of the file. Measured on the
# real file: memory 167 MB -> 70 MB, load time about 40% faster. This matters
# on Colab's limited RAM.
CITY_USECOLS = [
    "geoname_id", "city_name", "latitude", "longitude", "feature_class",
    "country_id", "admin1_code", "population", "timezone"
]

# zf.open() streams the .txt straight out of the zip; nothing is unpacked to disk.
with zipfile.ZipFile(city_zip_file) as zf:
    city_member = next(name for name in zf.namelist() if name.endswith(".txt"))
    cities = pd.read_csv(
        zf.open(city_member), sep="\t", header=None,
        names=city_columns, usecols=CITY_USECOLS,       # CHANGED: see CITY_USECOLS.
        # IDs are read as text so that codes with leading zeros, such as
        # admin1 "01", keep them.
        dtype={"geoname_id": str, "country_id": str, "admin1_code": str},
        low_memory=False,
        **NA_SAFE                                        # CHANGED: see NA_SAFE.
    )

# Feature class "P" = populated place (cities, towns, villages).
# CHANGED: filter before building new columns, so no work is spent on rows
# that are about to be removed.
cities = cities[cities["feature_class"] == "P"].copy()

# A city's region ID is "<country>.<admin1>", e.g. "US" + "." + "MI" = "US.MI".
# This is the same format as region_registry, so the two tables link up.
# CHANGED: replaced np.where(cond, str_series, np.nan), which mixed strings and
# float NaN and relied on numpy's type rules. Joining strings with NaN already
# gives NaN, and .where() states the rule directly.
cities["region_id"] = (cities["country_id"] + "." + cities["admin1_code"]).where(
    cities["admin1_code"].notna()
)
cities["geo_id"] = cities["geoname_id"]   # already text (dtype=str above)
cities["population"] = pd.to_numeric(cities["population"], errors="coerce")
cities = cities.dropna(subset=["country_id", "latitude", "longitude"])

# Rename everything to the shared layout: geo_id + place_name (+ parent IDs).
# drop_duplicates("geo_id") makes geo_id a unique key, so later merges can't
# duplicate rows.
country_registry = countries[[
    "ISO2", "ISO3", "country_name", "continent", "population"
]].rename(columns={
    "ISO2": "geo_id",
    "country_name": "place_name"
}).dropna(subset=["geo_id"]).drop_duplicates("geo_id")

region_registry = regions[[
    "region_id", "region_name", "country_id", "region_geoname_id"
]].rename(columns={
    "region_id": "geo_id",
    "region_name": "place_name"
}).drop_duplicates("geo_id")

city_registry = cities[[
    "geo_id", "city_name", "country_id", "region_id",
    "latitude", "longitude", "population", "timezone"
]].rename(columns={"city_name": "place_name"}).drop_duplicates("geo_id")

# Saved so you can look up the right geo_id when filling in the CSV templates.
country_registry.to_csv(REGISTRY_DIR / "countries.csv", index=False)
region_registry.to_csv(REGISTRY_DIR / "regions.csv", index=False)
city_registry.to_csv(REGISTRY_DIR / "cities_geonames1000.csv", index=False)

print("Countries/areas:", len(country_registry))
print("Administrative regions:", len(region_registry))
print("Populated-place records:", len(city_registry))

In [ ]:
# CELL 6 — SCREENING SETTINGS
# These numbers set how wide the funnel is. At most this many places advance:
#   30 countries x 5 regions x 5 cities = 750 cities.

TOP_COUNTRIES = 30
TOP_REGIONS_PER_COUNTRY = 5
TOP_CITIES_PER_REGION = 5

# Optional minimum score (0 to 1) on top of top-N. None = use only top-N.
MIN_COUNTRY_SCORE = None
MIN_REGION_SCORE = None
MIN_CITY_SCORE = None

# Minimum share of the total criterion weight a place must have data for.
# Example: at 0.60, a place with data covering only 40% of the weight would
# win or lose on too little evidence, so it gets no rank and does not advance.
MIN_DATA_COVERAGE = 0.60

# CHANGED: catch an impossible setting now, before it shows up as "0 advanced".
assert 0.0 <= MIN_DATA_COVERAGE <= 1.0, "MIN_DATA_COVERAGE must be between 0 and 1"

print("Screening settings loaded.")

In [ ]:
# CELL 7 — HELPER FUNCTIONS: CRITERIA, DATA LOADING, NORMALIZATION, ENTROPY WEIGHTS

def flatten_criteria(tier_name):
    "Return {criterion_key: metadata} for one tier, without the category level."
    return {
        key: details
        for category in CRITERIA[tier_name].values()
        for key, details in category.items()
    }

def load_tier_indicators(tier_name, registry):
    """
    Attach each populated criterion CSV to the registry as one column.
    Returns (data, loaded): data has one row per place and one column per
    loaded criterion (NaN where a place has no value); loaded lists those criteria.
    """
    data = registry.copy()
    data["geo_id"] = data["geo_id"].astype(str)
    registry_ids = set(data["geo_id"])
    metadata = flatten_criteria(tier_name)
    folder = TIER_FOLDERS[tier_name]
    indicator_columns = {}            # criterion_key -> Series indexed by geo_id

    for criterion_key in metadata:
        path = folder / f"{criterion_key}.csv"
        if not path.exists():
            continue

        # CHANGED: read geo_id as text with NA_SAFE. Before, "NA" (Namibia)
        # became NaN, and numeric city IDs in a column with any blank became
        # "123.0", which never matches "123". Either way the data was lost
        # without an error.
        indicator = pd.read_csv(path, dtype={"geo_id": str}, **NA_SAFE)
        required = {"geo_id", "value"}
        missing_columns = required - set(indicator.columns)
        if missing_columns:
            raise ValueError(f"{path.name} is missing columns: {missing_columns}")

        # CHANGED: strip stray spaces left by hand-edited spreadsheets (" US").
        indicator["geo_id"] = indicator["geo_id"].str.strip()
        # errors="coerce": text such as "n/a" becomes NaN instead of crashing.
        indicator[criterion_key] = pd.to_numeric(indicator["value"], errors="coerce")

        # CHANGED: drop rows with no usable value *before* removing duplicates,
        # so a blank first row can no longer win over a later row with data.
        indicator = indicator.dropna(subset=["geo_id", criterion_key])

        # A template that was never filled in is skipped, so only criteria
        # with real data take part in scoring.
        if indicator.empty:
            continue

        # CHANGED: if a place has several rows (several years), keep the newest
        # year instead of whichever row happens to come first.
        if "year" in indicator.columns:
            indicator["_year"] = pd.to_numeric(indicator["year"], errors="coerce")
            indicator = indicator.sort_values("_year", ascending=False, na_position="last")
        indicator = indicator.drop_duplicates("geo_id")

        # CHANGED: report IDs that match no place, so a typo in geo_id no
        # longer disappears silently. (In tiers 2-3 this also lists places
        # screened out earlier, which is expected.)
        unmatched = sorted(set(indicator["geo_id"]) - registry_ids)
        if unmatched:
            print(
                f"Note: {path.name}: {len(unmatched)} geo_id(s) not in this tier's "
                f"candidate set (typo, or screened out earlier). e.g. {unmatched[:5]}"
            )

        indicator_columns[criterion_key] = indicator.set_index("geo_id")[criterion_key]

    if not indicator_columns:
        raise ValueError(
            f"No populated criterion CSVs found for {tier_name}. "
            f"Fill at least one template in {folder} and rerun."
        )

    # CHANGED (optimization): one join for all criteria, instead of one merge
    # per criterion. Each merge copied the whole table, so this does less
    # work. A left join keeps every registry place, including those without data.
    data = data.join(pd.DataFrame(indicator_columns), on="geo_id")
    return data, list(indicator_columns)

def normalize_series(series, direction):
    """
    Min-max normalization: x' = (x - min) / (max - min), giving values in [0, 1].
    Cost criteria use 1 - x', so 1 is always the best place and 0 the worst.
    This puts criteria in different units (%, $, per 1,000) on one scale.
    Missing values stay missing; they are never treated as 0.
    """
    x = pd.to_numeric(series, errors="coerce")
    minimum, maximum = x.min(), x.max()     # pandas skips NaN when finding these

    if pd.isna(minimum) or pd.isna(maximum):
        return pd.Series(np.nan, index=x.index)

    if np.isclose(maximum, minimum):
        # Every place has the same value, so this criterion can't separate
        # them, and max - min = 0 would divide by zero. Give everyone 0.5.
        return pd.Series(0.5, index=x.index).where(x.notna())

    normalized = (x - minimum) / (maximum - minimum)
    if direction == "cost":
        normalized = 1 - normalized
    return normalized.clip(0, 1)            # guards against floating-point overshoot

def calculate_entropy_weights(normalized_df):
    """
    Entropy weight method (objective weighting). For criterion j over n places:
        p_ij = x_ij / sum_i x_ij               each place's share of the column
        e_j  = -(1 / ln n) * sum_i p_ij ln p_ij   entropy, between 0 and 1
        d_j  = 1 - e_j                         how much the criterion varies
        w_j  = d_j / sum_j d_j                 weights add up to 1
    If every place has the same value, e_j = 1, d_j = 0, and the weight is 0
    (no information). Criteria that spread places out unevenly get more weight.
    """
    X = normalized_df.copy().astype(float)

    # CHANGED: use only places with at least one observed value. Before, every
    # registry row went in (about 250 countries even when only 40 had data).
    # Places with no data were filled with column means, which pushed every
    # e_j toward 1, so the weights depended on registry size, not on the data.
    X = X.dropna(how="all")

    # A criterion needs at least 2 observations and 2 distinct values to carry information.
    usable = []
    for col in X.columns:
        observed = X[col].dropna()
        if len(observed) >= 2 and observed.nunique() > 1:
            usable.append(col)

    if not usable:
        if len(normalized_df.columns) == 0:
            return pd.Series(dtype=float)
        # No criterion separates the places, so equal weights is the neutral choice.
        return pd.Series(1 / len(normalized_df.columns), index=normalized_df.columns)

    X = X[usable].copy()
    # Fill gaps with the column mean (used for estimating weights only; scores
    # never use these filled values). This slightly lowers a criterion's spread
    # in proportion to its missing data, a fair penalty for patchy data.
    X = X.apply(lambda col: col.fillna(col.mean()), axis=0)
    # CHANGED: removed X.clip(lower=0), which never changed anything because
    # values are already in [0, 1].

    # min-max makes each column's worst place exactly 0, and ln(0) is undefined;
    # a tiny shift avoids it without changing the result in any meaningful way.
    P = X + 1e-12
    P = P.div(P.sum(axis=0), axis=1)        # p_ij: each column now sums to 1

    n = len(P)
    if n <= 1:
        # ln(1) = 0 would divide by zero, and one place can't be compared with anything.
        return pd.Series(1 / len(usable), index=usable).reindex(
            normalized_df.columns, fill_value=0.0
        )  # CHANGED: return every criterion (0 for unusable) so the result's shape is always the same.

    k = 1 / np.log(n)                       # scales e_j into [0, 1]
    entropy = -k * (P * np.log(P)).sum(axis=0)
    divergence = (1 - entropy).clip(lower=0)

    if divergence.sum() <= 1e-12:
        weights = pd.Series(1 / len(usable), index=usable)
    else:
        weights = divergence / divergence.sum()

    # Put the unusable criteria back with weight 0, so every criterion appears in the output.
    final = pd.Series(0.0, index=normalized_df.columns)
    final.loc[weights.index] = weights
    return final

In [ ]:
# CELL 8 — SCORE A TIER USING ENTROPY WEIGHTS

def score_tier(data, tier_name, loaded_criteria, fixed_weights=None,
               override=None, min_coverage=None):
    """
    Score every place in one tier.
        score_i    = sum_j(w_j * x'_ij) / sum_j(w_j), summing only over criteria place i has
        coverage_i = sum_j(w_j over criteria place i has) / sum_j(w_j)
    A missing value is skipped, not counted as 0, so missing data neither helps
    nor hurts the score. Coverage records how much evidence stands behind it.

    override: optional tuple (criterion_key, geo_id, replacement_raw_value).
    fixed_weights: optional Series of weights to hold constant (used in the sensitivity test).
    min_coverage: places below this coverage get a score but no rank.
        None = use MIN_DATA_COVERAGE, read when the function runs, so a changed
        setting takes effect without rerunning this cell.
    Returns (scored data sorted best-first, weights, normalized matrix).
    """
    if min_coverage is None:
        min_coverage = MIN_DATA_COVERAGE
    metadata = flatten_criteria(tier_name)
    work = data.copy()                      # never modify the caller's data

    if override is not None:
        criterion_key, target_geo_id, replacement_value = override
        if criterion_key not in loaded_criteria:
            raise ValueError(f"{criterion_key} is not loaded in {tier_name}.")
        mask = work["geo_id"].astype(str) == str(target_geo_id)
        if not mask.any():
            raise ValueError(f"geo_id {target_geo_id} was not found in this tier.")
        work.loc[mask, criterion_key] = float(replacement_value)

    # Normalization runs after the override on purpose: a test value outside the
    # old min/max stretches the scale for everyone, just as real new data would.
    normalized = pd.DataFrame(index=work.index)
    for key in loaded_criteria:
        normalized[key] = normalize_series(
            work[key], metadata[key]["direction"]
        )

    if fixed_weights is None:
        weights = calculate_entropy_weights(normalized)
    else:
        weights = fixed_weights.reindex(loaded_criteria).fillna(0.0)
        if weights.sum() <= 0:
            weights = pd.Series(1 / len(loaded_criteria), index=loaded_criteria)
        else:
            weights = weights / weights.sum()    # rescale so the weights add up to 1

    # Each value times its weight. NaN * w stays NaN, so missing data is skipped.
    weighted_values = normalized.mul(weights, axis=1)
    # Total weight of the criteria each place actually has data for.
    available_weight = normalized.notna().mul(weights, axis=1).sum(axis=1)
    # min_count=1: a place with no data gets NaN, not a fake score of 0.
    weighted_sum = weighted_values.sum(axis=1, min_count=1)

    # Divide by the available weight (not by 1) to rescale over the criteria the place has.
    work["mcda_score"] = weighted_sum / available_weight.replace(0, np.nan)
    work["data_coverage"] = available_weight / weights.sum()

    # CHANGED: rank only places that meet the coverage threshold. Before,
    # low-coverage places took up rank numbers and were then removed, which
    # left gaps (1, 2, 5, ...). method="min": tied places share the best rank.
    eligible = work["mcda_score"].notna() & (work["data_coverage"] >= min_coverage)
    work["rank"] = work["mcda_score"].where(eligible).rank(method="min", ascending=False)

    return work.sort_values("mcda_score", ascending=False), weights, normalized

def keep_top_n(data, n, minimum_score=None, group_column=None):
    """
    Keep the top N places overall (group_column=None), or the top N inside each
    parent (e.g. 5 regions per country). The per-parent version keeps the
    shortlist spread out, so one strong country can't take every slot.
    """
    result = data.dropna(subset=["mcda_score"]).copy()
    if minimum_score is not None:
        result = result[result["mcda_score"] >= minimum_score]
    if group_column is None:
        return result.nlargest(n, "mcda_score")
    # Sort once, best first; head(n) then takes each group's first n rows.
    return (
        result.sort_values("mcda_score", ascending=False)
        .groupby(group_column, group_keys=False)
        .head(n)
    )

# CHANGED: one shared check for CELLS 10-11. Before, an empty tier went
# straight into scoring and produced confusing empty tables.
def require_candidates(candidates, tier_label, parent_label):
    if candidates.empty:
        raise ValueError(
            f"No {tier_label} candidates: none of the selected {parent_label} have "
            f"{tier_label} entries in the registry. Check the earlier tier's results."
        )

In [ ]:
# CELL 9 — RUN TIER 1: COUNTRY SCREENING
# Score every country in the world, drop low-coverage ones, keep the top N.

country_data, tier1_loaded = load_tier_indicators(
    "tier1_country", country_registry
)
countries_scored, tier1_weights, tier1_normalized = score_tier(
    country_data, "tier1_country", tier1_loaded
)

countries_scored = countries_scored[
    countries_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_countries = keep_top_n(
    countries_scored,
    TOP_COUNTRIES,
    MIN_COUNTRY_SCORE
)

print("Tier 1 criteria loaded:", tier1_loaded)
print("Countries scored:", len(countries_scored))
print("Countries advanced:", len(selected_countries))
display(selected_countries[[
    "geo_id", "place_name", "mcda_score", "data_coverage", "rank"
]].head(20))
display(tier1_weights.rename("entropy_weight").to_frame())

In [ ]:
# CELL 10 — RUN TIER 2: REGION SCREENING WITHIN SELECTED COUNTRIES
# Only regions inside the selected countries are scored. Normalization and
# weights are therefore relative to this smaller pool, not to the whole world.

selected_country_ids = selected_countries["geo_id"].astype(str).tolist()

candidate_regions = region_registry[
    region_registry["country_id"].astype(str).isin(selected_country_ids)
].copy()
require_candidates(candidate_regions, "region", "countries")   # CHANGED: see helper.

region_data, tier2_loaded = load_tier_indicators(
    "tier2_region", candidate_regions
)
regions_scored, tier2_weights, tier2_normalized = score_tier(
    region_data, "tier2_region", tier2_loaded
)

regions_scored = regions_scored[
    regions_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_regions = keep_top_n(
    regions_scored,
    TOP_REGIONS_PER_COUNTRY,
    MIN_REGION_SCORE,
    group_column="country_id"
)

print("Tier 2 criteria loaded:", tier2_loaded)
print("Candidate regions:", len(candidate_regions))
print("Regions advanced:", len(selected_regions))

# CHANGED: name any selected country that has no region advancing. Before,
# such a country could win Tier 1 and then vanish without explanation.
dropped_countries = sorted(set(selected_country_ids) - set(selected_regions["country_id"]))
if dropped_countries:
    print("Selected countries with no region advancing (check Tier 2 data):", dropped_countries)

display(selected_regions[[
    "geo_id", "place_name", "country_id",
    "mcda_score", "data_coverage", "rank"
]].head(30))
display(tier2_weights.rename("entropy_weight").to_frame())

In [ ]:
# CELL 11 — RUN TIER 3: CITY SCREENING WITHIN SELECTED REGIONS
# Same pattern one level down. Note: "rank" compares each city with every
# candidate city across all regions; selection takes the top N per region.

selected_region_ids = selected_regions["geo_id"].astype(str).tolist()

candidate_cities = city_registry[
    city_registry["region_id"].astype(str).isin(selected_region_ids)
].copy()
require_candidates(candidate_cities, "city", "regions")        # CHANGED: see helper.

city_data, tier3_loaded = load_tier_indicators(
    "tier3_city", candidate_cities
)
cities_scored, tier3_weights, tier3_normalized = score_tier(
    city_data, "tier3_city", tier3_loaded
)

cities_scored = cities_scored[
    cities_scored["data_coverage"] >= MIN_DATA_COVERAGE
]

selected_cities = keep_top_n(
    cities_scored,
    TOP_CITIES_PER_REGION,
    MIN_CITY_SCORE,
    group_column="region_id"
)

print("Tier 3 criteria loaded:", tier3_loaded)
print("Candidate cities:", len(candidate_cities))
print("Cities advanced:", len(selected_cities))

# CHANGED: same check as Tier 2, for selected regions that lost every city.
dropped_regions = sorted(set(selected_region_ids) - set(selected_cities["region_id"]))
if dropped_regions:
    print("Selected regions with no city advancing (check Tier 3 data):", dropped_regions)

display(selected_cities[[
    "geo_id", "place_name", "country_id", "region_id",
    "population", "mcda_score", "data_coverage", "rank"
]].head(50))
display(tier3_weights.rename("entropy_weight").to_frame())

In [ ]:
# CELL 12 — EXPORT SCORES, SHORTLISTS, AND WEIGHTS
# "all_*_scores" = every place that passed the coverage filter;
# "selected_*"   = the shortlist that advanced to the next tier.

countries_scored.to_csv(OUTPUT_DIR / "tier1_all_country_scores.csv", index=False)
selected_countries.to_csv(OUTPUT_DIR / "tier1_selected_countries.csv", index=False)

regions_scored.to_csv(OUTPUT_DIR / "tier2_all_region_scores.csv", index=False)
selected_regions.to_csv(OUTPUT_DIR / "tier2_selected_regions.csv", index=False)

cities_scored.to_csv(OUTPUT_DIR / "tier3_all_city_scores.csv", index=False)
selected_cities.to_csv(OUTPUT_DIR / "tier3_selected_cities.csv", index=False)

# Each tier has its own criteria, so each column is filled only for that tier's
# rows and is empty (NaN) elsewhere.
pd.DataFrame({
    "tier1_country": tier1_weights,
    "tier2_region": tier2_weights,
    "tier3_city": tier3_weights
}).to_csv(OUTPUT_DIR / "entropy_weights_by_tier.csv")

print("All results exported to:", OUTPUT_DIR)

In [ ]:
# CELL 13 — USER-CONTROLLED SENSITIVITY SETTINGS
# Question answered: "If this place's value for this criterion were X instead,
# how would its score and rank change?" This shows how robust a ranking is.

SENSITIVITY_TIER = "tier3_city"             # tier1_country / tier2_region / tier3_city
SENSITIVITY_CRITERION = "restaurants_per_10000"
SENSITIVITY_GEO_ID = "YOUR_CITY_GEONAME_ID" # replace with an ID in the scored tier

# Replace these example values with plausible values in the criterion's real unit.
TEST_VALUES = [20, 30, 40, 50, 60]

print("Sensitivity settings ready. Update the three settings and run the next cell.")

In [ ]:
# CELL 14 — RUN THE SENSITIVITY TEST
# Weights stay fixed at the baseline entropy weights. Otherwise changing one
# value would also shift the weights, and the effect of the value change
# couldn't be told apart from the effect of the weight change.
# CHANGED: this cell appeared twice, word for word; the second copy was removed.

TIER_RESULTS = {
    "tier1_country": {
        "data": country_data, "loaded": tier1_loaded, "weights": tier1_weights
    },
    "tier2_region": {
        "data": region_data, "loaded": tier2_loaded, "weights": tier2_weights
    },
    "tier3_city": {
        "data": city_data, "loaded": tier3_loaded, "weights": tier3_weights
    }
}

if SENSITIVITY_TIER not in TIER_RESULTS:
    raise ValueError("SENSITIVITY_TIER must be tier1_country, tier2_region, or tier3_city.")

sens = TIER_RESULTS[SENSITIVITY_TIER]
if SENSITIVITY_CRITERION not in sens["loaded"]:
    raise ValueError(
        f"{SENSITIVITY_CRITERION} is not loaded. "
        f"Available: {sens['loaded']}"
    )

# CHANGED: the baseline uses the stored weights (sens["weights"]), the same
# fixed-weight path as the test runs, so the overridden value is the only
# difference between them. The original recalculated the weights from scratch,
# which was redundant.
base_scored, _, _ = score_tier(
    sens["data"], SENSITIVITY_TIER, sens["loaded"], fixed_weights=sens["weights"]
)
target_mask = base_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
target_base = base_scored[target_mask]
if target_base.empty:
    raise ValueError("Target ID not found in this tier's scored dataset.")
target_base = target_base.iloc[0]

# CHANGED: baseline_value used to be computed and then ignored. It is now a
# reference row and a red marker on both plots. A missing value gets a clear
# note instead of silently becoming NaN.
baseline_value = target_base[SENSITIVITY_CRITERION]
if pd.isna(baseline_value):
    print(f"Note: {SENSITIVITY_GEO_ID} has no observed {SENSITIVITY_CRITERION}; "
          "the tests below add a value where there was none.")
if pd.isna(target_base["rank"]):
    print(f"Note: {SENSITIVITY_GEO_ID} is below MIN_DATA_COVERAGE at baseline, so it has no rank.")

sensitivity_rows = [{
    "tier": SENSITIVITY_TIER,
    "geo_id": SENSITIVITY_GEO_ID,
    "criterion": SENSITIVITY_CRITERION,
    "scenario": "baseline",
    "tested_value": baseline_value,
    "mcda_score": target_base["mcda_score"],
    "rank": target_base["rank"],
    "data_coverage": target_base["data_coverage"]
}]
# Rerun the scoring once per test value. Every place is re-normalized each
# time, so the rank reflects the full effect of the change.
for test_value in TEST_VALUES:
    scenario_scored, _, _ = score_tier(
        sens["data"],
        SENSITIVITY_TIER,
        sens["loaded"],
        fixed_weights=sens["weights"],
        override=(SENSITIVITY_CRITERION, SENSITIVITY_GEO_ID, test_value)
    )
    target_row = scenario_scored[
        scenario_scored["geo_id"].astype(str) == str(SENSITIVITY_GEO_ID)
    ].iloc[0]

    sensitivity_rows.append({
        "tier": SENSITIVITY_TIER,
        "geo_id": SENSITIVITY_GEO_ID,
        "criterion": SENSITIVITY_CRITERION,
        "scenario": "test",
        "tested_value": test_value,
        "mcda_score": target_row["mcda_score"],
        "rank": target_row["rank"],
        "data_coverage": target_row["data_coverage"]
    })

sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

sensitivity_results.to_csv(
    OUTPUT_DIR / "sensitivity_analysis.csv", index=False
)

# CHANGED: one plotting function replaces two near-identical copy-pasted blocks.
tests = sensitivity_results[sensitivity_results["scenario"] == "test"].sort_values("tested_value")

def plot_sensitivity(column, ylabel, title, invert_y=False):
    "Line = tested values; red dot = where the place stands today."
    plt.figure(figsize=(8, 5))
    plt.plot(tests["tested_value"], tests[column], marker="o", label="tested values")
    if pd.notna(baseline_value):
        plt.scatter([baseline_value], [target_base[column]], color="red",
                    zorder=3, label="current (baseline)")
        plt.legend()
    plt.xlabel(f"{SENSITIVITY_CRITERION} (raw value)")
    plt.ylabel(ylabel)
    plt.title(title)
    if invert_y:
        plt.gca().invert_yaxis()             # rank 1 at the top reads as "higher is better"
    plt.grid(True)
    plt.show()

plot_sensitivity("mcda_score", "MCDA score",
                 f"Sensitivity of {SENSITIVITY_GEO_ID} to {SENSITIVITY_CRITERION}")
plot_sensitivity("rank", "Rank (1 = best)",
                 f"Rank sensitivity of {SENSITIVITY_GEO_ID}", invert_y=True)